# DP-filter candidate catalog

Per-tile lag-aware interval DP over rollout templates (`docs/dp_master` idea).
Emits ~72 handmade-format chains for the assignment master — no CP-SAT here.

In [1]:
import json
from dataclasses import dataclass
from pathlib import Path

HORIZON = 30  # calendar days 0..29
LAGS = (0, 1, 2)
CROP_PROFILE = "no_fert"
ANIMAL_PROFILE = "with_care"
BASE_CROPS = ("WHEAT", "CARROT")
EXTRA_CROPS = ("MELON", "TOMATO", "STRAWBERRY")
ANIMALS = ("GOOSE", "COW", "SHEEP")

DATA = Path("../data")
with (DATA / "crop_rollouts.json").open() as f:
    crops_data = json.load(f)
with (DATA / "animal_rollouts.json").open() as f:
    animals_data = json.load(f)


@dataclass(frozen=True)
class Template:
    name: str
    tile_free_age: int
    setup_cost: int
    base_price: int
    harvest_map: dict  # age -> units
    is_animal: bool

    def placement_value(self, start: int) -> int:
        value = -self.setup_cost
        for age, units in self.harvest_map.items():
            day = start + age
            if day < HORIZON:
                value += units * self.base_price
        return value

    def fits(self, start: int) -> bool:
        if self.is_animal:
            return start < HORIZON
        return start + self.tile_free_age < HORIZON

    @property
    def free_day(self) -> int:
        """Days after start until tile is free (for interval DP)."""
        return self.tile_free_age


def _crop_template(crop: str) -> Template:
    spec = crops_data["crops"][crop]
    prof = spec[CROP_PROFILE]
    harvest_map = dict(zip(prof["harvest_ages"], prof["yield_per_harvest"]))
    return Template(
        name=f"{crop}_{CROP_PROFILE}",
        tile_free_age=prof["tile_free_age"],
        setup_cost=spec["seed_cost"],
        base_price=spec["base_price"],
        harvest_map=harvest_map,
        is_animal=False,
    )


def _animal_template(animal: str) -> Template:
    spec = animals_data["animals"][animal]
    prof = spec[ANIMAL_PROFILE]
    harvest_map = dict(zip(prof["harvest_ages"], prof["yield_per_harvest"]))
    return Template(
        name=f"{animal}_{ANIMAL_PROFILE}",
        tile_free_age=HORIZON,  # occupies rest of season once placed
        setup_cost=spec["animal_cost"],
        base_price=spec["base_price"],
        harvest_map=harvest_map,
        is_animal=True,
    )


TEMPLATES = {t.name: t for t in [_crop_template(c) for c in BASE_CROPS + EXTRA_CROPS] + [_animal_template(a) for a in ANIMALS]}
TEMPLATES

{'WHEAT_no_fert': Template(name='WHEAT_no_fert', tile_free_age=4, setup_cost=10, base_price=25, harvest_map={4: 4}, is_animal=False),
 'CARROT_no_fert': Template(name='CARROT_no_fert', tile_free_age=3, setup_cost=20, base_price=35, harvest_map={3: 3}, is_animal=False),
 'MELON_no_fert': Template(name='MELON_no_fert', tile_free_age=10, setup_cost=80, base_price=250, harvest_map={10: 6}, is_animal=False),
 'TOMATO_no_fert': Template(name='TOMATO_no_fert', tile_free_age=11, setup_cost=50, base_price=60, harvest_map={8: 1, 9: 1, 10: 1, 11: 1}, is_animal=False),
 'STRAWBERRY_no_fert': Template(name='STRAWBERRY_no_fert', tile_free_age=16, setup_cost=100, base_price=120, harvest_map={10: 1, 12: 1, 14: 1, 16: 1}, is_animal=False),
 'GOOSE_with_care': Template(name='GOOSE_with_care', tile_free_age=30, setup_cost=300, base_price=50, harvest_map={4: 4, 5: 2, 6: 2, 7: 2, 8: 2, 9: 2, 10: 2, 11: 2, 12: 2, 13: 2, 14: 2, 15: 2, 16: 2, 17: 2, 18: 2, 19: 2, 20: 2, 21: 2, 22: 2, 23: 2, 24: 2, 25: 2, 26: 

In [2]:
def _placement(template: Template, start: int) -> dict:
    free_day = start + template.tile_free_age
    return {
        "name": template.name,
        "start": start,
        "free_day": free_day,
        "value": template.placement_value(start),
        "template": template,
    }


def generate_placements(templates: list[Template]) -> list[dict]:
    out = []
    for tmpl in templates:
        for start in range(HORIZON):
            if tmpl.fits(start):
                out.append(_placement(tmpl, start))
    return out


def weighted_interval_dp(candidates: list[dict], lag: int) -> tuple[int, list[dict]]:
    """Max-value non-overlapping set with seam rule: next.start >= prev.free_day + lag."""
    cands = sorted(candidates, key=lambda c: (c["start"], c["free_day"]))
    n = len(cands)
    pred = [-1] * n
    for i in range(n):
        lo, hi, best = 0, i - 1, -1
        while lo <= hi:
            mid = (lo + hi) // 2
            if cands[mid]["free_day"] + lag <= cands[i]["start"]:
                best = mid
                lo = mid + 1
            else:
                hi = mid - 1
        pred[i] = best

    dp = [0] * (n + 1)
    take = [False] * n
    for i in range(1, n + 1):
        idx = i - 1
        prev_val = dp[pred[idx] + 1] if pred[idx] >= 0 else 0
        incl = cands[idx]["value"] + prev_val
        excl = dp[i - 1]
        if incl >= excl:
            dp[i] = incl
            take[idx] = True
        else:
            dp[i] = excl

    selected, i = [], n
    while i > 0:
        idx = i - 1
        if take[idx]:
            selected.append(cands[idx])
            i = pred[idx] + 1 if pred[idx] >= 0 else 0
        else:
            i -= 1
    selected.reverse()
    return dp[n], selected


def placements_to_chain(selected: list[dict]) -> list[list]:
    return [[p["name"], p["start"]] for p in selected]


def chain_value(chain: list[list]) -> int:
    total = 0
    for name, start in chain:
        total += TEMPLATES[name].placement_value(start)
    return total


def _mono_greedy(template: Template, lag: int) -> list[list]:
    chain, start = [], 0
    while template.fits(start):
        chain.append([template.name, start])
        start = start + template.tile_free_age + lag
    return chain


def mono(crop: str, lag: int) -> list[list]:
    return _mono_greedy(TEMPLATES[f"{crop}_{CROP_PROFILE}"], lag)


def mix(lag: int) -> list[list]:
    templates = [TEMPLATES[f"{c}_{CROP_PROFILE}"] for c in BASE_CROPS]
    _, selected = weighted_interval_dp(generate_placements(templates), lag)
    return placements_to_chain(selected)

In [3]:
def _base_templates(base: str) -> list[Template]:
    if base == "mix":
        return [TEMPLATES[f"{c}_{CROP_PROFILE}"] for c in BASE_CROPS]
    return [TEMPLATES[f"{base}_{CROP_PROFILE}"]]


def _pack_window(templates: list[Template], lag: int, start_min: int, end_exclusive: int) -> list[list]:
    """Best base-crop chain in [start_min, end_exclusive) respecting seam before end_exclusive."""
    cands = [
        c
        for c in generate_placements(templates)
        if c["start"] >= start_min and c["free_day"] + lag <= end_exclusive
    ]
    _, selected = weighted_interval_dp(cands, lag)
    return placements_to_chain(selected)


def insert_crop(base: str, extra_crop: str, lag: int) -> list[list]:
    extra = TEMPLATES[f"{extra_crop}_{CROP_PROFILE}"]
    base_tmps = _base_templates(base)
    best_chain, best_val = [], -10**18

    for t_extra in range(HORIZON):
        if not extra.fits(t_extra):
            continue
        prefix = _pack_window(base_tmps, lag, 0, t_extra)
        suffix_start = t_extra + extra.tile_free_age + lag
        suffix = _pack_window(base_tmps, lag, suffix_start, HORIZON) if suffix_start < HORIZON else []
        chain = prefix + [[extra.name, t_extra]] + suffix
        val = chain_value(chain)
        if val > best_val:
            best_val, best_chain = val, chain
    return best_chain


def insert_animal(base: str, animal: str, lag: int) -> list[list]:
    animal_tmpl = TEMPLATES[f"{animal}_{ANIMAL_PROFILE}"]
    base_tmps = _base_templates(base)
    best_chain, best_val = [], -10**18

    for t in range(HORIZON):
        if not animal_tmpl.fits(t):
            continue
        prefix = _pack_window(base_tmps, lag, 0, t)
        chain = prefix + [[animal_tmpl.name, t]]
        val = chain_value(chain)
        if val > best_val:
            best_val, best_chain = val, chain
    return best_chain


def full_animal(animal: str, start: int) -> list[list]:
    name = f"{animal}_{ANIMAL_PROFILE}"
    return [[name, start]]

In [4]:
def build_catalog() -> list[dict]:
    entries = []

    def add(family: str, lag: int | None, chain: list[list], *, start: int | None = None):
        entries.append(
            {
                "family": family,
                "lag": lag,
                "start": start,
                "chain": chain,
                "value": chain_value(chain),
            }
        )

    for lag in LAGS:
        add("wheat_only", lag, mono("WHEAT", lag))
        add("carrot_only", lag, mono("CARROT", lag))
        add("wheat_carrot_mix", lag, mix(lag))

    for base in ("WHEAT", "CARROT", "mix"):
        base_label = {"WHEAT": "wheat_only", "CARROT": "carrot_only", "mix": "wheat_carrot_mix"}[base]
        for lag in LAGS:
            for extra in EXTRA_CROPS:
                add(f"{base_label}+{extra.lower()}", lag, insert_crop(base, extra, lag))
            for animal in ANIMALS:
                add(f"{base_label}+{animal.lower()}", lag, insert_animal(base, animal, lag))

    for animal in ANIMALS:
        for start in LAGS:
            add(f"full_{animal.lower()}", None, full_animal(animal, start), start=start)

    return entries


catalog = build_catalog()
handmade_chains = [e["chain"] for e in catalog] + [[]]
print(f"{len(catalog)} DP chains + 1 idle -> {len(handmade_chains)} catalog entries")

72 DP chains + 1 idle -> 73 catalog entries


In [5]:
# Compact table
print(f"{'family':<28} {'lag/start':>9} {'value':>7}  placements")
print("-" * 90)
for e in catalog:
    lag_s = str(e["start"]) if e["start"] is not None else str(e["lag"])
    pl = ", ".join(f"{n}@{d}" for n, d in e["chain"])
    print(f"{e['family']:<28} {lag_s:>9} {e['value']:>7}  {pl}")

family                       lag/start   value  placements
------------------------------------------------------------------------------------------
wheat_only                           0     630  WHEAT_no_fert@0, WHEAT_no_fert@4, WHEAT_no_fert@8, WHEAT_no_fert@12, WHEAT_no_fert@16, WHEAT_no_fert@20, WHEAT_no_fert@24
carrot_only                          0     765  CARROT_no_fert@0, CARROT_no_fert@3, CARROT_no_fert@6, CARROT_no_fert@9, CARROT_no_fert@12, CARROT_no_fert@15, CARROT_no_fert@18, CARROT_no_fert@21, CARROT_no_fert@24
wheat_carrot_mix                     0     775  WHEAT_no_fert@0, WHEAT_no_fert@4, CARROT_no_fert@8, CARROT_no_fert@11, CARROT_no_fert@14, CARROT_no_fert@17, CARROT_no_fert@20, CARROT_no_fert@23, CARROT_no_fert@26
wheat_only                           1     540  WHEAT_no_fert@0, WHEAT_no_fert@5, WHEAT_no_fert@10, WHEAT_no_fert@15, WHEAT_no_fert@20, WHEAT_no_fert@25
carrot_only                          1     595  CARROT_no_fert@0, CARROT_no_fert@4, CARROT_no_fert@8

In [6]:
def starts_for(family: str, lag: int) -> list[int]:
    for e in catalog:
        if e["family"] == family and e["lag"] == lag:
            return [d for _, d in e["chain"]]
    return []


print("=== lag seam sanity (wheat-only starts) ===")
for lag in LAGS:
    starts = starts_for("wheat_only", lag)
    print(f"lag {lag}: {starts}")

# lag-0 same-day replant: consecutive wheat cycles share calendar day
wheat_lag0 = next(e["chain"] for e in catalog if e["family"] == "wheat_only" and e["lag"] == 0)
for i in range(len(wheat_lag0) - 1):
    prev_start = wheat_lag0[i][1]
    next_start = wheat_lag0[i + 1][1]
    free_day = prev_start + TEMPLATES["WHEAT_no_fert"].tile_free_age
    assert free_day == next_start, (prev_start, free_day, next_start)
print("lag-0 wheat: harvest/plant same day OK")

print("\n=== exactly-one extra checks ===")
for extra in EXTRA_CROPS:
    prof = f"{extra}_{CROP_PROFILE}"
    for base in ("wheat_only", "carrot_only", "wheat_carrot_mix"):
        for lag in LAGS:
            fam = f"{base}+{extra.lower()}"
            chain = next(e["chain"] for e in catalog if e["family"] == fam and e["lag"] == lag)
            count = sum(1 for n, _ in chain if n == prof)
            assert count == 1, (fam, lag, count)

for animal in ANIMALS:
    prof = f"{animal}_{ANIMAL_PROFILE}"
    for base in ("wheat_only", "carrot_only", "wheat_carrot_mix"):
        for lag in LAGS:
            fam = f"{base}+{animal.lower()}"
            chain = next(e["chain"] for e in catalog if e["family"] == fam and e["lag"] == lag)
            count = sum(1 for n, _ in chain if n == prof)
            assert count == 1, (fam, lag, count)

print("each extra crop/animal appears exactly once per insert chain")

print("\n=== full animal (start delay 0-2) ===")
for e in catalog:
    if e["family"].startswith("full_"):
        print(e["family"], "start", e["start"], e["chain"], "value", e["value"])

=== lag seam sanity (wheat-only starts) ===
lag 0: [0, 4, 8, 12, 16, 20, 24]
lag 1: [0, 5, 10, 15, 20, 25]
lag 2: [0, 6, 12, 18, 24]
lag-0 wheat: harvest/plant same day OK

=== exactly-one extra checks ===
each extra crop/animal appears exactly once per insert chain

=== full animal (start delay 0-2) ===
full_goose start 0 [['GOOSE_with_care', 0]] value 2400
full_goose start 1 [['GOOSE_with_care', 1]] value 2300
full_goose start 2 [['GOOSE_with_care', 2]] value 2200
full_cow start 0 [['COW_with_care', 0]] value 5360
full_cow start 1 [['COW_with_care', 1]] value 5360
full_cow start 2 [['COW_with_care', 2]] value 4880
full_sheep start 0 [['SHEEP_with_care', 0]] value 6300
full_sheep start 1 [['SHEEP_with_care', 1]] value 6300
full_sheep start 2 [['SHEEP_with_care', 2]] value 6300


In [7]:
candidates = [e["chain"] for e in catalog] + [[]]
print(len(candidates), "candidates (72 DP + 1 idle)")
candidates

73 candidates (72 DP + 1 idle)


[[['WHEAT_no_fert', 0],
  ['WHEAT_no_fert', 4],
  ['WHEAT_no_fert', 8],
  ['WHEAT_no_fert', 12],
  ['WHEAT_no_fert', 16],
  ['WHEAT_no_fert', 20],
  ['WHEAT_no_fert', 24]],
 [['CARROT_no_fert', 0],
  ['CARROT_no_fert', 3],
  ['CARROT_no_fert', 6],
  ['CARROT_no_fert', 9],
  ['CARROT_no_fert', 12],
  ['CARROT_no_fert', 15],
  ['CARROT_no_fert', 18],
  ['CARROT_no_fert', 21],
  ['CARROT_no_fert', 24]],
 [['WHEAT_no_fert', 0],
  ['WHEAT_no_fert', 4],
  ['CARROT_no_fert', 8],
  ['CARROT_no_fert', 11],
  ['CARROT_no_fert', 14],
  ['CARROT_no_fert', 17],
  ['CARROT_no_fert', 20],
  ['CARROT_no_fert', 23],
  ['CARROT_no_fert', 26]],
 [['WHEAT_no_fert', 0],
  ['WHEAT_no_fert', 5],
  ['WHEAT_no_fert', 10],
  ['WHEAT_no_fert', 15],
  ['WHEAT_no_fert', 20],
  ['WHEAT_no_fert', 25]],
 [['CARROT_no_fert', 0],
  ['CARROT_no_fert', 4],
  ['CARROT_no_fert', 8],
  ['CARROT_no_fert', 12],
  ['CARROT_no_fert', 16],
  ['CARROT_no_fert', 20],
  ['CARROT_no_fert', 24]],
 [['WHEAT_no_fert', 0],
  ['WHEAT_no_